In [0]:
# create a spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("trading_sentiment_platform").getOrCreate()

In [0]:
#
from pyspark.sql import functions as F

top100_comp_10Ks = spark\
                    .read\
                    .table("workspace.sec_filings.top100_comp_sec_filings")\
                    .filter(F.col("form_type") == "10-K")

display(top100_comp_10Ks)

### Pull html

##### Get texts

##### Extract specific sections

##### Get each section by file

In [0]:
import requests
from bs4 import BeautifulSoup
import re

headers = {"User-Agent": "Joel Doh joeljuniordoh19@gmail.com"}


# ------------------------------------------------------------
# 1. REMOVE XBRL + CLEAN TEXT
# ------------------------------------------------------------
def clean_inline_xbrl(html):
    soup = BeautifulSoup(html, "html.parser")

    # Remove inline XBRL tags
    for ix in soup.find_all(re.compile("^ix:")):
        ix.unwrap()

    # Remove scripts, styles, tables
    for tag in soup(["script", "style", "table"]):
        tag.decompose()

    txt = soup.get_text(" ", strip=True)
    txt = re.sub(r"\s+", " ", txt)
    txt = txt.replace("\u00A0", " ")

    return txt


# ------------------------------------------------------------
# 2. SUPER-ROBUST NORMALIZATION OF HEADINGS
# ------------------------------------------------------------
def normalize_headers(text):
    t = text.upper()

    # Allow *any* amount of spacing inside letters: B U S I N E S S
    def loose(word):
        return "".join([ch + r"\s*" for ch in word])

    BUSINESS = loose("BUSINESS")
    RISKFACT = loose("RISKFACTORS")
    MGMT = loose("MANAGEMENT")
    DISC = loose("DISCUSSION")
    ANAL = loose("ANALYSIS")

    # Fix ITEM 1
    t = re.sub(
        rf"ITEM\s*1\s*[\.\-]?\s*{BUSINESS}",
        "ITEM 1. BUSINESS",
        t
    )

    # Fix ITEM 1A
    t = re.sub(
        rf"ITEM\s*1A\s*[\.\-]?\s*{RISKFACT}",
        "ITEM 1A. RISK FACTORS",
        t
    )

    # Fix ITEM 7 MD&A
    t = re.sub(
        rf"ITEM\s*7\s*[\.\-]?\s*{MGMT}[^\n]{{0,80}}?{DISC}[^\n]{{0,80}}?{ANAL}",
        "ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS",
        t
    )

    return t


# ------------------------------------------------------------
# 3. MAIN PARSER
# ------------------------------------------------------------
def parse_main_filing(filing_url):
    try:
        print(f"\nDownloading: {filing_url}")
        html = requests.get(filing_url, headers=headers, timeout=15).text

        # Clean XBRL
        cleaned = clean_inline_xbrl(html)

        # Normalize headings even if split
        norm = normalize_headers(cleaned)

        u = norm

        # ------------------------------------------------------------------
        # ITEM 1 — BUSINESS
        # ------------------------------------------------------------------
        m = re.search(
            r"ITEM 1\. BUSINESS(.*?)(?=ITEM 1A\. RISK FACTORS|ITEM 1B\.|ITEM 2\.)",
            u,
            re.DOTALL
        )
        business = m.group(1).strip() if m else ""

        # ------------------------------------------------------------------
        # ITEM 1A — RISK FACTORS
        # ------------------------------------------------------------------
        m = re.search(
            r"ITEM 1A\. RISK FACTORS(.*?)(?=ITEM 1B\.|ITEM 2\.|ITEM 3\.)",
            u,
            re.DOTALL
        )
        risk = m.group(1).strip() if m else ""

        # ------------------------------------------------------------------
        # ITEM 7 — MD&A
        # ------------------------------------------------------------------
        m = re.search(
            r"ITEM 7\. MANAGEMENT’S DISCUSSION AND ANALYSIS(.*?)(?=ITEM 7A\.|ITEM 8\.|PART III)",
            u,
            re.DOTALL
        )
        mda = m.group(1).strip() if m else ""

        return u, business, risk, mda

    except Exception as e:
        print("Error:", e)
        return "", "", "", ""


# ------------------------------------------------------------
# TEST WITH MICROSOFT 2025
# ------------------------------------------------------------
# url = "https://www.sec.gov/Archives/edgar/data/200406/000020040625000038/jnj-20241229.htm"
# t, business, risk, mda = parse_main_filing(url)

# print("FULL TEXT:", len(t))
# print("TEXT:", t[10000:30000])
# print("\nBUSINESS:", len(business))
# print("BUSINESS_TEXT:", business[:500])
# print("\nRISK:", len(risk))
# print("RISK_text:", risk[:500])
# print("\nMDA:", len(mda))
# print("MDA_TEXT:", mda[:500])


In [0]:
# Import PySpark UDF tools and functions
from pyspark.sql.functions import udf
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# ----------------------------------------------------------
# Create a UDF that wraps the 'parsing' function
# The UDF returns a struct with 3 string fields:
#   - cleaned_text
#   - risks_text
#   - mda_text
# ----------------------------------------------------------
parse_udf = udf(
    parse_main_filing,
    StructType([
        StructField("cleaned_text", StringType()),
        StructField("bus_text", StringType()),
        StructField("risks_text", StringType()),
        StructField("mda_text", StringType())
    ])
)

# ----------------------------------------------------------
# Apply the UDF to each filing_url row in top100_raw_files
# This creates a new column "parsed" that is a struct
# containing the output of parsing():
#   parsed.cleaned_text
#   parsed.risks_text
#   parsed.mda_text
# ----------------------------------------------------------
parsed_df = (
    top100_comp_10Ks
        .where(F.col("company_name")!="COSTCO WHOLESALE CORP /NEW")
        # Apply UDF to filing_url
        .withColumn("parsed", parse_udf(F.col("filing_url")))

        # Select original metadata + extracted NLP text
        .select(
            "cik",
            "company_name",
            "tickers",
            "form_type",
            "filing_date",
            "accessionNumber",
            "market_cap",
            "sector",
            "filing_url",

            # Extract fields inside struct "parsed"
            F.col("parsed.cleaned_text").alias("text"),
            F.col("parsed.bus_text").alias("bus_text"),
            F.col("parsed.risks_text").alias("risks_text"),
            F.col("parsed.mda_text").alias("mda_text")
        )
)
display(parsed_df)


In [0]:
parsed_df_n = parsed_df.select(
    "cik",
    "company_name",
    "tickers",
    "filing_date",
    "accessionNumber",
    "market_cap",
    "sector",
    "filing_url",
    "bus_text",
    "risks_text",
    "mda_text"
)

display(parsed_df_n)


In [0]:
parsed_df_n.printSchema()

In [0]:
parsed_df_n.write.format("delta").mode("overwrite").saveAsTable("workspace.sec_filings.filings_10K_text")

In [0]:
from pyspark.sql import functions as F

spark.sql("SHOW TABLES IN workspace.sec_filings").display()

In [0]:
spark.sql("DROP TABLE IF EXISTS sec_filings.comp_10ks;")
spark.sql("DROP TABLE IF EXISTS sec_filings.costco_10k_text;")